# Scratch – repr & visualisation testing

Temporary notebook for testing `_repr_html_` / `__repr__` across all classes and visualisation helpers like `coverage_bar()` and `tree()`.

**Delete when done.**

In [ ]:
import sys
if "google.colab" in sys.modules:
    %pip install -q git+https://github.com/rebase-energy/TimeDataModel.git

In [1]:
from datetime import datetime, timezone

import numpy as np

from timedatamodel import (
    AggregationMethod,
    CoverageBar,
    DataType,
    Dimension,
    Frequency,
    GeoLocation,
    HierarchicalTimeSeries,
    HierarchyNode,
    TimeSeriesList,
    TimeSeriesArray,
    TimeSeriesCollection,
    TimeSeriesTable,
    TimeSeriesType,
    get_theme,
    set_theme,
    reset_theme,
)

## Theme colors

All repr and visualisation functions read from a shared theme (`theme.json`). Use `get_theme()` to inspect the current colors and `set_theme()` to override them. Changes take effect on the next repr call. Call `reset_theme()` to restore defaults.

In [2]:
# Override colors here — uncomment and edit any line to experiment.
# Changes apply to all reprs rendered after this cell.

set_theme({
    "light": {
        "header_bg": "#f0f0f0",
        "header_text": "#1a1a1a",
        "header_border": "#4a4a4a",
        "meta_bg": "#fafafa",
        "meta_label": "#475569",
        "meta_value": "#1a1a1a",
        "col_header_text": "#555",
        "col_header_border": "#ccc",
        "data_text": "#1a1a1a",
        "index_text": "#1e293b",
        "hover_bg": "#f5f5f5",
        "ellipsis": "#999",
        "coverage_present": "#4CAF50",
        "coverage_absent": "#e0e0e0",
        "coverage_label": "#333",
        "coverage_date": "#666",
    },
    "dark": {
        "header_bg": "#1e293b",
        "header_text": "#e2e8f0",
        "header_border": "#475569",
        "meta_bg": "#0f172a",
        "meta_label": "#94a3b8",
        "meta_value": "#e2e8f0",
        "col_header_text": "#94a3b8",
        "col_header_border": "#334155",
        "data_text": "#e2e8f0",
        "index_text": "#cbd5e1",
        "hover_bg": "#1e293b",
        "ellipsis": "#64748b",
    },
})

# To restore defaults at any point:
# reset_theme()

## Helper – generate hourly timestamps

In [3]:
def hourly_ts(n=24, start="2024-01-15"):
    base = datetime.fromisoformat(f"{start}T00:00:00+00:00")
    from datetime import timedelta
    return [base + timedelta(hours=i) for i in range(n)]

---
## 1. TimeSeriesList

In [4]:
ts = TimeSeriesList(
    Frequency.PT1H,
    timezone="UTC",
    timestamps=hourly_ts(24),
    values=[120 + 5 * np.sin(i) for i in range(24)],
    name="power",
    unit="MW",
    data_type=DataType.MEASUREMENT,
    description="Hourly power output from wind farm Alpha",
)
ts

Name,power
Length,24
Frequency,PT1H
Timezone,UTC (+00:00)
Unit,MW
Data type,MEASUREMENT
Description,Hourly power output from wind farm Alpha
timestamp,power
2024-01-15 00:00,120.0
2024-01-15 01:00,124.207
2024-01-15 02:00,124.546


In [5]:
print(ts)

TimeSeriesList
┌──────────────────────────────────┐
│  Name:             power         │
│  Length:           24            │
│  Frequency:        PT1H          │
│  Timezone:         UTC (+00:00)  │
│  Unit:             MW            │
│  Data type:        MEASUREMENT   │
├──────────────────────────────────┤
│  2024-01-15 00:00    120.0       │
│  2024-01-15 01:00  124.207       │
│  2024-01-15 02:00  124.546       │
│  ...                   ...       │
│  2024-01-15 21:00  124.183       │
│  2024-01-15 22:00  119.956       │
│  2024-01-15 23:00  115.769       │
└──────────────────────────────────┘


### TimeSeriesList – empty

In [6]:
ts_empty = TimeSeriesList(Frequency.PT1H, timestamps=[], values=[], name="empty")
ts_empty

TimeSeriesList
┌───────────────────────────┐
│  Name:             empty  │
│  Length:           0      │
│  Frequency:        PT1H   │
│  Timezone:         UTC    │
├───────────────────────────┤
│  (empty)                  │
└───────────────────────────┘

### TimeSeriesList – with NaN gaps

In [7]:
vals_gap = [float(i) for i in range(24)]
for i in [5, 6, 7, 15, 16]:
    vals_gap[i] = None

ts_gap = TimeSeriesList(
    Frequency.PT1H,
    timestamps=hourly_ts(24),
    values=vals_gap,
    name="gappy_signal",
    unit="kW",
)
ts_gap

Name,gappy_signal
Length,24
Frequency,PT1H
Timezone,UTC (+00:00)
Unit,kW
timestamp,gappy_signal
2024-01-15 00:00,0.0
2024-01-15 01:00,1.0
2024-01-15 02:00,2.0
…,…
2024-01-15 21:00,21.0


### TimeSeriesList – short (no truncation)

In [8]:
ts_short = TimeSeriesList(
    Frequency.PT1H,
    timestamps=hourly_ts(5),
    values=[1.0, 2.0, 3.0, 4.0, 5.0],
    name="short",
)
ts_short

Name,short
Length,5
Frequency,PT1H
Timezone,UTC (+00:00)
timestamp,short
2024-01-15 00:00,1.0
2024-01-15 01:00,2.0
2024-01-15 02:00,3.0


### TimeSeriesList – with location and labels

In [9]:
ts_loc = TimeSeriesList(
    Frequency.PT1H,
    timestamps=hourly_ts(24),
    values=[100 + i * 0.5 for i in range(24)],
    name="temperature",
    unit="°C",
    location=GeoLocation(latitude=59.91, longitude=10.75),
    labels={"source": "met.no", "station": "Oslo-Blindern"},
    data_type=DataType.MEASUREMENT,
    description="Air temperature at Blindern station",
)
ts_loc

Name,temperature
Length,24
Frequency,PT1H
Timezone,UTC (+00:00)
Unit,°C
Data type,MEASUREMENT
Location,"59.91°N, 10.75°E"
Description,Air temperature at Blindern station
Labels,"{'source': 'met.no', 'station': 'Oslo-Blindern'}"
timestamp,temperature
2024-01-15 00:00,100.0


---
## 2. TimeSeriesTable

In [10]:
timestamps = hourly_ts(24)
tbl = TimeSeriesTable(
    Frequency.PT1H,
    timezone="UTC",
    timestamps=timestamps,
    values=np.column_stack([
        [120 + 5 * np.sin(i) for i in range(24)],
        [80 + 3 * np.cos(i) for i in range(24)],
        [200 + 10 * np.sin(i + 1) for i in range(24)],
    ]),
    names=["wind_farm_A", "wind_farm_B", "solar_park_C"],
    units=["MW", "MW", "MW"],
    data_types=[DataType.MEASUREMENT, DataType.MEASUREMENT, DataType.FORECAST],
)
tbl

TimeSeriesTable
┌────────────────────────────────────────────────────────────┐
│  Name:             unnamed                                 │
│  Columns:          wind_farm_A, wind_farm_B, solar_park_C  │
│  Length:           24 × 3                                  │
│  Frequency:        PT1H                                    │
│  Timezone:         UTC (+00:00)                            │
│  Unit:             MW, MW, MW                              │
│  Data type:        MEASUREMENT, MEASUREMENT, FORECAST      │
├────────────────────────────────────────────────────────────┤
│                    wind_farm_A  wind_farm_B  solar_park_C  │
│  2024-01-15 00:00        120.0         83.0       208.415  │
│  2024-01-15 01:00      124.207      81.6209       209.093  │
│  2024-01-15 02:00      124.546      78.7516       201.411  │
│  ...                       ...          ...           ...  │
│  2024-01-15 21:00      124.183      78.3568       199.911  │
│  2024-01-15 22:00      119.956      77.0001       191.538  │
│  2024-01-15 23:00      115.769      78.4015       190.944  │
└────────────────────────────────────────────────────────────┘

In [11]:
print(tbl)

TimeSeriesTable
┌────────────────────────────────────────────────────────────┐
│  Name:             unnamed                                 │
│  Columns:          wind_farm_A, wind_farm_B, solar_park_C  │
│  Length:           24 × 3                                  │
│  Frequency:        PT1H                                    │
│  Timezone:         UTC (+00:00)                            │
│  Unit:             MW, MW, MW                              │
│  Data type:        MEASUREMENT, MEASUREMENT, FORECAST      │
├────────────────────────────────────────────────────────────┤
│                    wind_farm_A  wind_farm_B  solar_park_C  │
│  2024-01-15 00:00        120.0         83.0       208.415  │
│  2024-01-15 01:00      124.207      81.6209       209.093  │
│  2024-01-15 02:00      124.546      78.7516       201.411  │
│  ...                       ...          ...           ...  │
│  2024-01-15 21:00      124.183      78.3568       199.911  │
│  2024-01-15 22:00      119.956      7

---
## 3. TimeSeriesCollection

In [12]:
ts_a = TimeSeriesList(
    Frequency.PT1H, timestamps=hourly_ts(24), values=[float(i) for i in range(24)],
    name="series_A", unit="MW",
)
ts_b = TimeSeriesList(
    Frequency.P1D,
    timestamps=[datetime.fromisoformat(f"2024-01-{d:02d}T00:00:00+00:00") for d in range(1, 8)],
    values=[100.0, 105.0, 98.0, None, 110.0, 115.0, 108.0],
    name="series_B", unit="GWh",
)

coll = TimeSeriesCollection(
    [ts_a, ts_b, tbl],
    name="My Collection",
    description="A mix of different series",
)
coll

name,type,freq,tz,length,begin,end
series_A,TimeSeriesList,PT1H,UTC,24,2024-01-15 00:00,2024-01-15 23:00
series_B,TimeSeriesList,P1D,UTC,7,2024-01-01 00:00,2024-01-07 00:00
"wind_farm_A,wind_farm_B,solar_park_C",TimeSeriesTable,PT1H,UTC,24,2024-01-15 00:00,2024-01-15 23:00


In [13]:
print(coll)

TimeSeriesCollection
┌────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│  name                                  type             freq  tz   length  begin             end               │
├────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│  series_A                              TimeSeriesList   PT1H  UTC  24      2024-01-15 00:00  2024-01-15 23:00  │
│  series_B                              TimeSeriesList   P1D   UTC  7       2024-01-01 00:00  2024-01-07 00:00  │
│  wind_farm_A,wind_farm_B,solar_park_C  TimeSeriesTable  PT1H  UTC  24      2024-01-15 00:00  2024-01-15 23:00  │
└────────────────────────────────────────────────────────────────────────────────────────────────────────────────┘


---
## 4. TimeSeriesArray (N-dimensional)

In [14]:
time_labels = hourly_ts(12)
scenarios = ["low", "mid", "high"]

arr = TimeSeriesArray(
    Frequency.PT1H,
    name="price_scenarios",
    unit="EUR/MWh",
    data_type=DataType.SCENARIO,
    dimensions=[
        Dimension(name="timestamp", labels=time_labels),
        Dimension(name="scenario", labels=scenarios),
    ],
    values=np.random.default_rng(42).uniform(30, 80, size=(12, 3)),
)
arr

TimeSeriesArray
┌────────────────────────────────────────────────┐
│  Name:             price_scenarios             │
│  Dimensions:       timestamp: 12, scenario: 3  │
│  Shape:            (12, 3)                     │
│  Frequency:        PT1H                        │
│  Timezone:         UTC (+00:00)                │
│  Unit:             EUR/MWh                     │
│  Data type:        SCENARIO                    │
└────────────────────────────────────────────────┘

In [15]:
print(arr)

TimeSeriesArray
┌────────────────────────────────────────────────┐
│  Name:             price_scenarios             │
│  Dimensions:       timestamp: 12, scenario: 3  │
│  Shape:            (12, 3)                     │
│  Frequency:        PT1H                        │
│  Timezone:         UTC (+00:00)                │
│  Unit:             EUR/MWh                     │
│  Data type:        SCENARIO                    │
└────────────────────────────────────────────────┘


### TimeSeriesArray – 1D

In [16]:
arr_1d = TimeSeriesArray(
    Frequency.PT1H,
    name="load",
    unit="MW",
    dimensions=[Dimension(name="timestamp", labels=hourly_ts(8))],
    values=np.array([100, 105, 110, 108, 103, 99, 95, 92], dtype=float),
)
arr_1d

Name,load
Dimensions,timestamp: 8
Shape,"(8,)"
Frequency,PT1H
Timezone,UTC (+00:00)
Unit,MW
timestamp,load
2024-01-15 00:00:00+00:00,100.0
2024-01-15 01:00:00+00:00,105.0
2024-01-15 02:00:00+00:00,110.0
…,…


---
## 5. CoverageBar

In [17]:
ts_gap.coverage_bar()

gappy_signal  █████░░░███████░░███████
              2024-01-15 00:00  2024-01-15 23:00

In [18]:
print(ts_gap.coverage_bar())

gappy_signal  █████░░░███████░░███████
              2024-01-15 00:00  2024-01-15 23:00


In [19]:
tbl.coverage_bar()

wind_farm_A   ████████████████████████
wind_farm_B   ████████████████████████
solar_park_C  ████████████████████████
              2024-01-15 00:00  2024-01-15 23:00

In [20]:
print(tbl.coverage_bar())

wind_farm_A   ████████████████████████
wind_farm_B   ████████████████████████
solar_park_C  ████████████████████████
              2024-01-15 00:00  2024-01-15 23:00


---
## 6. HierarchicalTimeSeries & tree()

In [21]:
timestamps = hourly_ts(24)
rng = np.random.default_rng(0)

def make_ts(name):
    return TimeSeriesList(
        Frequency.PT1H, timestamps=timestamps,
        values=rng.uniform(50, 200, 24).tolist(),
        name=name, unit="MW", data_type=DataType.MEASUREMENT,
    )

tree_dict = {
    "total": {
        "Norway": {"Bergen": "bergen", "Oslo": "oslo", "Trondheim": "trondheim"},
        "Sweden": {"Stockholm": "stockholm", "Gothenburg": "gothenburg"},
    }
}
series_map = {
    "bergen": make_ts("Bergen"),
    "oslo": make_ts("Oslo"),
    "trondheim": make_ts("Trondheim"),
    "stockholm": make_ts("Stockholm"),
    "gothenburg": make_ts("Gothenburg"),
}

hts = HierarchicalTimeSeries.from_dict(
    tree_dict, series_map,
    levels=["region", "country", "city"],
    name="Nordic Wind Power",
    description="Hierarchical wind power across Nordic cities",
    aggregation=AggregationMethod.SUM,
)
hts

HierarchicalTimeSeries
┌───────────────────────────────────────────────────────────────────┐
│  Name:             Nordic Wind Power                              │
│  Levels:           region, country, city                          │
│  Nodes:            16 (5 leaves)                                  │
│  Frequency:        PT1H                                           │
│  Timezone:         UTC (+00:00)                                   │
│  Unit:             MW                                             │
│  Aggregation:      sum                                            │
├───────────────────────────────────────────────────────────────────┤
│  name        level    length  begin             end               │
├───────────────────────────────────────────────────────────────────┤
│  Bergen      country  24      2024-01-15 00:00  2024-01-15 23:00  │
│  Oslo        country  24      2024-01-15 00:00  2024-01-15 23:00  │
│  Trondheim   country  24      2024-01-15 00:00  2024-01-15 23:00  │
│  Stockholm   country  24      2024-01-15 00:00  2024-01-15 23:00  │
│  Gothenburg  country  24      2024-01-15 00:00  2024-01-15 23:00  │
└───────────────────────────────────────────────────────────────────┘

In [23]:
print(hts)

HierarchicalTimeSeries
┌───────────────────────────────────────────────────────────────────┐
│  Name:             Nordic Wind Power                              │
│  Levels:           region, country, city                          │
│  Nodes:            16 (5 leaves)                                  │
│  Frequency:        PT1H                                           │
│  Timezone:         UTC (+00:00)                                   │
│  Unit:             MW                                             │
│  Aggregation:      sum                                            │
├───────────────────────────────────────────────────────────────────┤
│  name        level    length  begin             end               │
├───────────────────────────────────────────────────────────────────┤
│  Bergen      country  24      2024-01-15 00:00  2024-01-15 23:00  │
│  Oslo        country  24      2024-01-15 00:00  2024-01-15 23:00  │
│  Trondheim   country  24      2024-01-15 00:00  2024-01-15 23:00 

### HierarchyTree – HTML

In [24]:
hts.tree()

└── total (region)
    └── total (region)
        ├── Norway (region)
        │   └── Norway (region)
        │       ├── Bergen (region)
        │       │   └── Bergen [24 pts]
        │       ├── Oslo (region)
        │       │   └── Oslo [24 pts]
        │       └── Trondheim (region)
        │           └── Trondheim [24 pts]
        └── Sweden (region)
            └── Sweden (region)
                ├── Stockholm (region)
                │   └── Stockholm [24 pts]
                └── Gothenburg (region)
                    └── Gothenburg [24 pts]

In [25]:
print(hts.tree())

└── total (region)
    └── total (region)
        ├── Norway (region)
        │   └── Norway (region)
        │       ├── Bergen (region)
        │       │   └── Bergen [24 pts]
        │       ├── Oslo (region)
        │       │   └── Oslo [24 pts]
        │       └── Trondheim (region)
        │           └── Trondheim [24 pts]
        └── Sweden (region)
            └── Sweden (region)
                ├── Stockholm (region)
                │   └── Stockholm [24 pts]
                └── Gothenburg (region)
                    └── Gothenburg [24 pts]


---
## 7. Wide TimeSeriesTable (many columns)

Test whether the repr truncates columns when there are too many to fit.

In [26]:
n_cols = 20
timestamps = hourly_ts(48)
rng = np.random.default_rng(7)

tbl_wide = TimeSeriesTable(
    Frequency.PT1H,
    timezone="UTC",
    timestamps=timestamps,
    values=rng.uniform(50, 300, size=(48, n_cols)),
    names=[f"turbine_{i:02d}" for i in range(n_cols)],
    units=["MW"] * n_cols,
)
tbl_wide

TimeSeriesTable
┌────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│  Name:             unnamed                                                                                                                                                                                                                                         │
│  Columns:          turbine_00, turbine_01, turbine_02, turbine_03, turbine_04, turbine_05, turbine_06, turbine_07, turbine_08, turbine_09, turbine_10, turbine_11, turbine_12, turbine_13, turbine_14, turbine_15, turbine_16, turbine_17, turbine_18, turbine_19  │
│  Length:           48 × 20                                                                                                                                                                                                                                         │
│  Frequency:        PT1H                                                                                                                                                                                                                                            │
│  Timezone:         UTC (+00:00)                                                                                                                                                                                                                                    │
│  Unit:             MW, MW, MW, MW, MW, MW, MW, MW, MW, MW, MW, MW, MW, MW, MW, MW, MW, MW, MW, MW                                                                                                                                                                  │
├────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│                    turbine_00  turbine_01  turbine_02  turbine_03  turbine_04  turbine_05  turbine_06  turbine_07  turbine_08  turbine_09  turbine_10  turbine_11  turbine_12  turbine_13  turbine_14  turbine_15  turbine_16  turbine_17  turbine_18  turbine_19  │
│  2024-01-15 00:00     206.274     274.303     243.921     106.302     125.042     268.388     51.3163     255.307     249.267     166.984     125.758     119.606     113.717     161.269     176.137     188.374     298.875     248.165     205.545      297.24  │
│  2024-01-15 01:00     103.827      90.053     203.135     60.9855     58.9201     178.722     166.552     279.292     207.307     178.529     174.218     111.879     52.9485     98.1005     223.008     100.152     142.384     50.9336     257.512     88.6153  │
│  2024-01-15 02:00       116.9     270.083     177.448     261.788     209.929     235.443     72.8739     185.286     176.943     267.835     140.316     199.546     64.8129     146.908     130.759     87.5499     254.085     144.862     294.687     197.498  │
│  ...                      ...         ...         ...         ...         ...         ...         ...         ...         ...         ...         ...         ...         ...         ...         ...         ...         ...         ...         ...         ...  │
│  2024-01-16 21:00     115.077      195.91     288.058     254.283     210.182     179.705      53.783     272.679     66.7135     82.2703     223.268     253.036     246.638     163.101     94.2639     58.8256     262.845     294.424     111.034     266.519  │
│  2024-01-16 22:00     150.633     127.836     176.289      52.042     160.679     99.8887     228.114     175.643     267.213     274.162      129.19     120.096     284.783     175.999     74.1289     61.4713     297.606     194.791     63.6232     177.814  │
│  2024-01-16 23:00     176.026     158

In [27]:
print(tbl_wide)

TimeSeriesTable
┌────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│  Name:             unnamed                                                                                                                                                                                                                                         │
│  Columns:          turbine_00, turbine_01, turbine_02, turbine_03, turbine_04, turbine_05, turbine_06, turbine_07, turbine_08, turbine_09, turbine_10, turbine_11, turbine_12, turbine_13, turbine_14, turbine_15, turbine_16, turbine_17, turbine_18, turbine_19  │
│  Length:           48 × 20                                                                                                                                                                       

### Same data as pandas DataFrame (for comparison)

In [28]:
tbl_wide.df

,turbine_00,turbine_01,turbine_02,turbine_03,turbine_04,turbine_05,turbine_06,turbine_07,turbine_08,turbine_09,turbine_10,turbine_11,turbine_12,turbine_13,turbine_14,turbine_15,turbine_16,turbine_17,turbine_18,turbine_19
timestamp,,,,,,,,,,,,,,,,,,,,
2024-01-15 00:00:00+00:00,206.273867,274.303450,243.921423,106.301797,125.041571,268.388361,51.316326,255.307105,249.267357,166.983738,125.758107,119.606403,113.717397,161.269076,176.137065,188.374338,298.875071,248.165480,205.544807,297.240037
2024-01-15 01:00:00+00:00,103.827175,90.053008,203.134901,60.985502,58.920070,178.722205,166.551506,279.291943,207.306564,178.529412,174.218359,111.878731,52.948506,98.100536,223.008030,100.151681,142.384078,50.933561,257.511932,88.615270
2024-01-15 02:00:00+00:00,116.899826,270.083038,177.447702,261.787562,209.929292,235.442737,72.873901,185.285955,176.943059,267.834844,140.316015,199.546017,64.812911,146.907950,130.759087,87.549932,254.084526,144.861543,294.686971,197.497923
2024-01-15 03:00:00+00:00,201.264063,209.499145,219.112561,87.697005,160.078367,109.890990,150.624575,74.176023,291.957013,103.751009,217.941291,125.105020,268.519257,215.553685,82.903954,261.268580,286.237043,275.979197,192.429787,86.364988
2024-01-15 04:00:00+00:00,98.115874,281.976421,188.081622,95.138125,271.014224,210.392926,192.423569,144.071959,152.738821,109.872303,59.514322,269.054702,166.932554,186.908800,130.540827,237.831230,56.299218,143.046318,57.587574,80.723026
2024-01-15 05:00:00+00:00,291.787059,214.440183,157.055062,180.935027,268.202302,136.052667,197.572746,220.921093,138.853444,179.774622,241.311846,277.294828,87.765570,283.354848,51.294716,238.244376,252.631708,84.191013,154.725913,253.814069
2024-01-15 06:00:00+00:00,53.567797,207.115487,248.255915,178.250896,231.462355,106.605871,99.630287,140.781738,94.851507,136.515360,287.031015,193.333179,135.017017,117.881155,288.009873,161.119552,295.098688,178.880667,180.291532,274.135131
2024-01-15 07:00:00+00:00,235.691855,195.163216,156.662376,269.546965,152.911537,280.689894,67.178838,157.499215,179.878705,287.734549,112.749812,251.509787,219.117800,229.271476,207.405546,292.890177,133.170365,149.568890,100.727938,62.676014
2024-01-15 08:00:00+00:00,103.227049,278.866099,260.042204,78.101435,200.944756,169.799124,198.671218,214.818752,126.664873,290.337711,166.460009,207.025214,208.806546,95.972349,65.466355,152.879206,241.007522,253.805447,232.497312,78.301238


---
## 8. Side-by-side: light vs dark

Quick way to preview both colour schemes without changing your OS setting — wrap the HTML repr in a `<div>` that forces a colour scheme.

In [29]:
from IPython.display import HTML

light = f'<div style="color-scheme: light; padding: 10px;">{ts._repr_html_()}</div>'
dark = f'<div style="color-scheme: dark; background: #0f172a; padding: 10px; border-radius: 8px;">{ts._repr_html_()}</div>'

HTML(f"<h4>Light mode</h4>{light}<br><h4>Dark mode</h4>{dark}")

Name,power
Length,24
Frequency,PT1H
Timezone,UTC (+00:00)
Unit,MW
Data type,MEASUREMENT
Description,Hourly power output from wind farm Alpha
timestamp,power
2024-01-15 00:00,120.0
2024-01-15 01:00,124.207
2024-01-15 02:00,124.546
